# Replication Notebook for Table 1 Figure 2 and Figure 3

This notebook reproduces Table 1, Figure 2, and Figure 3 from the revision analysis. It uses the anonymized CSV files in this folder and writes the outputs to the `results` folder.

## 0 Import packages and define paths

In [3]:
%matplotlib inline

from pathlib import Path
import os
import csv
import warnings

os.environ.setdefault("MPLCONFIGDIR", "/private/tmp/matplotlib")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy.stats import ttest_ind
from IPython.display import display, Image

CODE_DIR = Path.cwd()
DATA_FILE = CODE_DIR / "PNAS_revision_regression_upload_data.csv"
EVENT_FILE = CODE_DIR / "Fig3_event_study_data.csv"
HET_FILE = CODE_DIR / "Fig4_did_topic_data.csv"
OUTDIR = CODE_DIR / "results"
OUTDIR.mkdir(parents=True, exist_ok=True)

for path in [DATA_FILE, EVENT_FILE, HET_FILE]:
    if not path.exists():
        raise FileNotFoundError(f"Required input file not found: {path}")

df = pd.read_csv(DATA_FILE, encoding="utf-8", low_memory=False)

sns.set_style("whitegrid")
plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 22,
    "axes.labelsize": 24,
    "axes.titlesize": 26,
    "axes.titleweight": "bold",
    "legend.fontsize": 20,
    "xtick.labelsize": 20,
    "ytick.labelsize": 20,
    "figure.dpi": 300,
    "savefig.dpi": 500,
})

print(f"Data loaded: {df.shape[0]:,} rows x {df.shape[1]:,} columns")
print(f"Output directory: {OUTDIR}")

Data loaded: 139,530 rows x 23 columns


## 1 Table 1 descriptive statistics

In [6]:
required_columns = [
    "author_id", "year", "treat",
    "log_us_federal_amount", "log_us_amount", "log_all_amount", "topic_change",
    "career_age", "log_paper_count", "log_cf", "log_all_us_collaborators_count",
]
missing_columns = [column for column in required_columns if column not in df.columns]
if missing_columns:
    raise ValueError(f"Missing columns for Table 1: {missing_columns}")

test_columns = [
    "log_us_federal_amount",
    "log_us_amount",
    "log_all_amount",
    "topic_change",
    "career_age",
    "log_paper_count",
    "log_cf",
    "log_all_us_collaborators_count",
]

test_titles = [
    "U.S. Federal Funding",
    "U.S. Funding",
    "Total Funding",
    "Pivot Size",
    "Career Age",
    "# Papers",
    "Average Cf",
    "# U.S. Collaborators",
]

def ttest_period(data, start_year, end_year):
    period = data.loc[data["year"].between(start_year, end_year)].copy()
    author_means = (
        period.groupby("author_id")[test_columns + ["treat"]]
        .mean(numeric_only=True)
        .reset_index()
    )
    rows = []
    for variable, title in zip(test_columns, test_titles):
        treated = author_means.loc[author_means["treat"] == 1, variable].dropna()
        control = author_means.loc[author_means["treat"] == 0, variable].dropna()
        statistic, pvalue = ttest_ind(treated, control, equal_var=False, nan_policy="omit")
        rows.append({
            "Period": f"{start_year}-{end_year}",
            "Variable": title,
            "Mean (Treatment)": treated.mean(),
            "Mean (Control)": control.mean(),
            "Difference": treated.mean() - control.mean(),
            "t statistic": statistic,
            "p value": pvalue,
            "N (Treatment)": treated.size,
            "N (Control)": control.size,
        })
    return rows

table1 = pd.DataFrame(
    ttest_period(df, 2014, 2018) +
    ttest_period(df, 2019, 2023)
)

table1_path = OUTDIR / "Table1_descriptive_statistics.xlsx"
table1.to_excel(table1_path, index=False, float_format="%.4f")

#print(f"Saved: {table1_path}")
display(table1.style.format({
    "Mean (Treatment)": "{:.4f}",
    "Mean (Control)": "{:.4f}",
    "Difference": "{:.4f}",
    "t statistic": "{:.3f}",
    "p value": "{:.4f}",
}))

,Period,Variable,Mean (Treatment),Mean (Control),Difference,t statistic,p value,N (Treatment),N (Control)
0,2014-2018,U.S. Federal Funding,8.4982,8.5226,-0.0244,-0.247,0.8046,4651,9302
1,2014-2018,U.S. Funding,8.5264,8.5499,-0.0235,-0.238,0.8122,4651,9302
2,2014-2018,Total Funding,8.7330,8.7246,0.0083,0.087,0.9308,4651,9302
3,2014-2018,Pivot Size,0.5149,0.5146,0.0003,0.125,0.9003,4651,9302
4,2014-2018,Career Age,21.0686,21.0173,0.0513,0.296,0.7674,4651,9302
5,2014-2018,# Papers,1.7906,1.7816,0.0090,0.790,0.4294,4651,9302
6,2014-2018,Average Cf,0.8090,0.8045,0.0045,0.820,0.4120,4651,9302
7,2014-2018,# U.S. Collaborators,2.6760,2.6932,-0.0172,-0.922,0.3566,4651,9302
8,2019-2023,U.S. Federal Funding,7.5109,7.8075,-0.2966,-2.903,0.0037,4651,9302
9,2019-2023,U.S. Funding,7.5711,7.8746,-0.3034,-2.980,0.0029,4651,9302


## 2 Figure 2 event study

In [8]:
event = pd.read_csv(EVENT_FILE)
expected_columns = {"outcome", "panel", "year", "estimate", "se", "ci_low", "ci_high"}
missing_columns = expected_columns.difference(event.columns)
if missing_columns:
    raise ValueError(f"Event-study file is missing columns: {sorted(missing_columns)}")

outcomes = ["Federal Funding", "U.S. Funding", "Total Funding", "Pivot Size"]
event_titles = ["U.S. Federal Funding", "U.S. Funding", "Total Funding", "Pivot Size"]
event_colors = ["#87bed0", "#d7828c", "#f39c7e", "#40c2a8"]

fig, axes = plt.subplots(2, 2, figsize=(18, 12))
for ax, outcome, title, color in zip(axes.flat, outcomes, event_titles, event_colors):
    values = event.loc[event["outcome"] == outcome].sort_values("year")
    if values.empty:
        raise ValueError(f"No rows found for outcome '{outcome}' in {EVENT_FILE}")
    ax.errorbar(
        values["year"],
        values["estimate"],
        yerr=[values["estimate"] - values["ci_low"], values["ci_high"] - values["estimate"]],
        fmt="o",
        color=color,
        markersize=9,
        linewidth=3,
        capsize=4,
        capthick=4,
    )
    ax.axhline(0, color="gray", linestyle="--", alpha=0.6)
    ax.axvline(2018.5, color="gray", linestyle="--", alpha=0.7)
    ax.set_title(title, fontweight="bold")
    ax.set_xticks(range(2014, 2024), labels=range(2014, 2024), rotation=45)
    ax.set_ylabel("Estimate ± 95% CI")
    ax.yaxis.set_major_locator(mticker.MaxNLocator(5))
    ax.grid(False)
    ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout()

figure2_path = OUTDIR / "Figure2_event_study.png"
fig.savefig(figure2_path, dpi=500, bbox_inches="tight", facecolor="white")
plt.close(fig)

#print(f"Saved: {figure2_path}")
display(Image(filename=str(figure2_path), width=1100))

## 3 Figure 3 heterogeneity analysis

In [10]:
def clean_esttab_cell(value):
    if value is None:
        return ""
    text = str(value).strip()
    if text.startswith('="') and text.endswith('"'):
        text = text[2:-1]
    elif text.startswith("="):
        text = text[1:]
    return text.strip('"')

def parse_float(value):
    text = clean_esttab_cell(value)
    if text == "":
        return np.nan
    return float(text)

def parse_heterogeneity_csv(path):
    with path.open("r", encoding="utf-8-sig", newline="") as handle:
        rows = [[clean_esttab_cell(cell) for cell in row] for row in csv.reader(handle)]
    title_row = rows[0]
    coef_row = next(row for row in rows if row and clean_esttab_cell(row[0]) == "did_topic")
    records = []
    column_index = 1
    while column_index + 3 < len(coef_row):
        title = clean_esttab_cell(title_row[column_index])
        if title:
            records.append({
                "model_title": title,
                "coef": parse_float(coef_row[column_index]),
                "p": parse_float(coef_row[column_index + 1]),
                "ci_low": parse_float(coef_row[column_index + 2]),
                "ci_high": parse_float(coef_row[column_index + 3]),
            })
        column_index += 4
    parsed = pd.DataFrame(records)
    if parsed.empty:
        raise ValueError("No heterogeneity estimates were parsed.")
    return parsed

heterogeneity = parse_heterogeneity_csv(HET_FILE)

dimension_labels = [
    ["High List-Related\nFields", "Low List-Related\nFields"],
    ["Ethnically Chinese\nScientists", "Non Ethnically\nChinese Scientists"],
    ["Junior", "Senior"],
]
column_titles = ["Field", "Name-inferred Ethnicity", "Career Age"]

def plot_two_estimates(ax, values, labels, title):
    x = np.array([0.2, 0.8])
    colors = ["#f39c7e", "#40c2a8"]
    span = max(values["ci_high"].max() - values["ci_low"].min(), 0.01)
    y_bottom = min(values["ci_low"].min(), 0) - 0.12 * span
    y_top = max(values["ci_high"].max(), 0) + 0.30 * span

    for j, (_, row) in enumerate(values.reset_index(drop=True).iterrows()):
        ax.errorbar(
            x[j], row["coef"],
            yerr=[[row["coef"] - row["ci_low"]], [row["ci_high"] - row["coef"]]],
            fmt="o", capsize=15, capthick=5.5, color=colors[j], markersize=17, linewidth=5.5,
        )
        if row["p"] < 0.001:
            stars = "***"
        elif row["p"] < 0.05:
            stars = "**"
        elif row["p"] < 0.10:
            stars = "*"
        else:
            stars = "n.s."
        ax.text(
            x[j], row["ci_high"] + 0.045 * span, stars,
            ha="center", va="bottom", color=colors[j], fontweight="bold", fontsize=30,
        )

    ax.axhline(0, color="gray", linestyle="--", linewidth=1.2)
    ax.set_ylim(y_bottom, y_top)
    ax.set_xlim(0, 1)
    ax.set_xticks(x)
    ax.set_xticklabels(labels, fontsize=23, fontweight="bold")
    ax.tick_params(axis="y", labelsize=23)
    ax.tick_params(axis="x", length=0, pad=8)
    ax.set_title(title, fontsize=29, fontweight="bold", pad=12)
    ax.grid(False)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["left"].set_visible(False)
    ax.spines["bottom"].set_color("#BFBFBF")
    ax.spines["bottom"].set_linewidth(1)

fig, axes = plt.subplots(2, 3, figsize=(22, 13), dpi=400)
dimension_order = [1, 0, 2]

for outcome_index, outcome_title in enumerate(["U.S. Federal Funding", "U.S. Funding"]):
    for plot_column, original_dimension_index in enumerate(dimension_order):
        start = outcome_index * 6 + original_dimension_index * 2
        plot_two_estimates(
            axes[outcome_index, plot_column],
            heterogeneity.iloc[start:start + 2],
            dimension_labels[plot_column],
            outcome_title,
        )
    axes[outcome_index, 0].set_ylabel(
        r"Coef. $DID \times Pivot\ Size$", fontsize=28, fontweight="normal", labelpad=18,
    )

plt.subplots_adjust(left=0.08, right=0.99, top=0.88, bottom=0.09, wspace=0.23, hspace=0.42)
for col_index, col_title in enumerate(column_titles):
    pos = axes[0, col_index].get_position()
    x_center = (pos.x0 + pos.x1) / 2
    fig.text(x_center, 0.965, col_title, ha="center", va="center", fontsize=32, fontweight="bold")

figure3_path = OUTDIR / "Figure3_heterogeneity.png"
fig.savefig(figure3_path, dpi=500, bbox_inches="tight", facecolor="white")
plt.close(fig)

#print(f"Saved: {figure3_path}")
display(heterogeneity)
display(Image(filename=str(figure3_path), width=1400))

,model_title,coef,p,ci_low,ci_high
0,Federal: Ethnically Chinese scientists,0.584,0.254,-0.420,1.588
1,Federal: Non-ethnically Chinese scientists,0.779,0.021,0.118,1.440
2,Federal: High list-related fields,-0.134,0.773,-1.046,0.777
3,Federal: Low list-related fields,1.148,0.001,0.488,1.807
4,Federal: Junior scientists,0.940,0.023,0.130,1.751
5,Federal: Senior scientists,0.210,0.550,-0.478,0.897
6,U.S.: Ethnically Chinese scientists,0.637,0.211,-0.361,1.635
7,U.S.: Non-ethnically Chinese scientists,0.826,0.014,0.167,1.485
8,U.S.: High list-related fields,-0.091,0.842,-0.993,0.810
9,U.S.: Low list-related fields,1.185,0.000,0.525,1.845


## 4 Output files

In [13]:
for path in sorted(OUTDIR.iterdir()):
    if path.is_file():
        print(f"{path.name:45s} {path.stat().st_size / 1024:10.1f} KB")

Figure2_event_study.png                            839.4 KB
Figure3_heterogeneity.png                         1137.6 KB
Table1_descriptive_statistics.xlsx                   5.8 KB
Table2_baseline_DID_10_columns.rtf                  10.7 KB
Table3_moderating_effects_6_columns.rtf              8.3 KB
